# PPR Wetland Mapping Experiment

**Multi-Temporal NAIP + LiDAR Wetland Mapping with Weakly Supervised Deep Learning**

Authors: Jayakumar Pujar, Qiusheng Wu

This notebook runs the full experiment pipeline on Google Colab:
1. **Data Download** -- NAIP (2015, 2017), 3DEP DEM, NWI
2. **Composites & Weak Labels** -- Spectral indices, depression extraction, label generation
3. **Model Training** -- U-Net++ and DeepLabV3+ with focal loss
4. **Inference & Evaluation** -- Prediction, dynamics mapping, accuracy assessment

**Runtime:** Use GPU (Runtime > Change runtime type > T4 GPU)

**Estimated time:** ~2-4 hours for full pipeline (data download + 50 epochs training)

---
## 0. Environment Setup

In [ ]:
# Verify GPU is available
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be very slow.")
    print("Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# Clone the research project repository
!git clone https://github.com/jayakumarpujar/wetlands-mapping-with-geoai.git
%cd wetlands-mapping-with-geoai

In [ ]:
# Install dependencies (torch with CUDA for Colab GPU)
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q geoai-py segmentation-models-pytorch rasterio geopandas scikit-learn py3dep

In [ ]:
# Verify imports
from research_paper.wetland import (
    PPR_STUDY_AREA,
    EXPERIMENT_DEFAULTS,
    COWARDIN_CLASSES,
    build_experiment_config,
    format_results_table,
)

print("Study area:", PPR_STUDY_AREA["name"])
print("Bbox:", PPR_STUDY_AREA["bbox"])
print("NAIP years:", PPR_STUDY_AREA["naip_years"])
print("Classes:", COWARDIN_CLASSES)
print("\nDefault config:")
for k, v in EXPERIMENT_DEFAULTS.items():
    if k != "architectures":
        print(f"  {k}: {v}")
print("  architectures:")
for arch in EXPERIMENT_DEFAULTS["architectures"]:
    print(f"    - {arch['architecture']} ({arch['encoder_name']})")
print("\nAll imports successful.")

---
## 1. Data Setup (Google Drive — Recommended)

The 3DEP and STAC APIs frequently time out in Colab. Upload pre-downloaded data instead.

### DEM Tiles
**Download from [USGS National Map](https://apps.nationalmap.gov/downloader/):**
- Elevation Products > 1/3 arc-second DEM > Current > GeoTIFF
- Bbox: Xmin=-100.55, Ymin=46.65, Xmax=-99.15, Ymax=47.60
- 4 tiles: n47w100, n47w101, n48w100, n48w101 (~1.5 GB total)

### NAIP Imagery
**Download from [NRCS Data Gateway](https://datagateway.nrcs.usda.gov/GDGOrder.aspx):**
- Custom Order > Select State (ND/SD) > NAIP DOQQ > Select years (2015, 2017)
- Or download from [USGS EarthExplorer](https://earthexplorer.usgs.gov/)
- Organize in year subfolders: `naip/2015/*.tif`, `naip/2017/*.tif`

**Upload to Google Drive**, then mount below.

In [ ]:
import os
import glob
from pathlib import Path

# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

# === DEM Tiles ===
DEM_DRIVE_DIR = "/content/drive/MyDrive/wetlands_data/dem_tiles"  # <-- UPDATE THIS PATH
DEM_TILES_DIR = "/content/dem_tiles"
os.makedirs(DEM_TILES_DIR, exist_ok=True)

import shutil
for name in os.listdir(DEM_DRIVE_DIR):
    if name.endswith(".tif"):
        src = os.path.join(DEM_DRIVE_DIR, name)
        dst = os.path.join(DEM_TILES_DIR, name)
        if not os.path.exists(dst):
            print(f"Copying {name} ...")
            shutil.copy2(src, dst)
        else:
            print(f"Already exists: {name}")

DEM_TILE_PATHS = sorted(Path(DEM_TILES_DIR).glob("USGS_13_*.tif"))
print(f"\nFound {len(DEM_TILE_PATHS)} DEM tiles:")
for p in DEM_TILE_PATHS:
    print(f"  {p} ({p.stat().st_size / 1e6:.1f} MB)")

# === NAIP Imagery ===
NAIP_DRIVE_DIR = "/content/drive/MyDrive/wetlands_data/naip"  # <-- UPDATE THIS PATH

# Collect NAIP files from year subfolders (naip/2015/*.tif, naip/2017/*.tif)
NAIP_2015 = sorted(glob.glob(os.path.join(NAIP_DRIVE_DIR, "2015", "*.tif")))
NAIP_2017 = sorted(glob.glob(os.path.join(NAIP_DRIVE_DIR, "2017", "*.tif")))

print(f"\nFound {len(NAIP_2015)} NAIP files for 2015")
print(f"Found {len(NAIP_2017)} NAIP files for 2017")

if not NAIP_2015 and not NAIP_2017:
    # Fallback: check if files are directly in naip/ without year subfolders
    all_naip = sorted(glob.glob(os.path.join(NAIP_DRIVE_DIR, "*.tif")))
    if all_naip:
        print(f"\nNo year subfolders found, but found {len(all_naip)} TIF files in {NAIP_DRIVE_DIR}")
        print("Please organize into year subfolders: naip/2015/*.tif, naip/2017/*.tif")
        # Show what's there to help organize
        for f in all_naip[:5]:
            print(f"  {os.path.basename(f)}")
        if len(all_naip) > 5:
            print(f"  ... and {len(all_naip) - 5} more")
    else:
        print(f"\nWARNING: No NAIP files found in {NAIP_DRIVE_DIR}")
        print("Upload NAIP TIF files to Google Drive in year subfolders.")

In [ ]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(name)s %(levelname)s %(message)s",
)

from research_paper.run_experiment import run_ppr_experiment

# Quick smoke test -- just 2 epochs to verify the pipeline
# Uses pre-downloaded DEM tiles and NAIP to skip flaky APIs
smoke_results = run_ppr_experiment(
    output_root="/content/ppr_smoke_test",
    overrides={
        "num_epochs": 2,
        "batch_size": 2,
        "pre_downloaded_dem_tiles": [str(p) for p in DEM_TILE_PATHS],
        "pre_downloaded_naip": {
            2015: NAIP_2015,
            2017: NAIP_2017,
        },
    },
)

print("Smoke test complete!")
print(f"Results saved to: {smoke_results['output_path']}")

---
## 2. Baseline Experiment (50 epochs)

Full training run with default hyperparameters. Trains both U-Net++ and DeepLabV3+ with ResNet-50 encoder.

In [ ]:
# Full baseline experiment
baseline_results = run_ppr_experiment(
    output_root="/content/ppr_baseline",
    overrides={
        "num_epochs": 50,
        "batch_size": 8,
        "pre_downloaded_dem_tiles": [str(p) for p in DEM_TILE_PATHS],
        "pre_downloaded_naip": {
            2015: NAIP_2015,
            2017: NAIP_2017,
        },
    },
)

print("\nBaseline experiment complete!")

In [ ]:
# Display baseline results
table = format_results_table(baseline_results["results"])
print(table)

---
## 3. Hyperparameter Sweep

Test different learning rates to find the optimal configuration.

In [ ]:
# Learning rate sweep
lr_results = {}
for lr in [1e-4, 5e-4, 1e-3]:
    print(f"\n{'='*60}")
    print(f"Training with learning_rate={lr}")
    print(f"{'='*60}")

    result = run_ppr_experiment(
        output_root=f"/content/ppr_lr_{lr}",
        overrides={
            "num_epochs": 50,
            "batch_size": 8,
            "learning_rate": lr,
            "pre_downloaded_dem_tiles": [str(p) for p in DEM_TILE_PATHS],
            "pre_downloaded_naip": {
                2015: NAIP_2015,
                2017: NAIP_2017,
            },
        },
    )
    lr_results[lr] = result["results"]

# Compare results across learning rates
print("\n" + "="*60)
print("LEARNING RATE COMPARISON")
print("="*60)
for lr, results in lr_results.items():
    print(f"\nlr={lr}:")
    for r in results:
        oa = r.get("overall_accuracy", None)
        miou = r.get("mean_iou", None)
        oa_str = f"{oa:.4f}" if oa is not None else "N/A"
        miou_str = f"{miou:.4f}" if miou is not None else "N/A"
        print(f"  {r['name']}: OA={oa_str}, mIoU={miou_str}")

---
## 4. Ablation Study

Test the contribution of different input features by removing them one at a time.

In [ ]:
# Ablation: Single architecture (faster), vary input channels
# Note: Ablation requires modifying the composite creation to exclude specific bands.
# This cell provides the framework -- uncomment and adapt based on your needs.

ablation_configs = [
    {
        "name": "Full model (10 channels)",
        "in_channels": 10,
        "description": "NAIP(4) + NDVI(2) + NDWI(2) + Elevation + Depression",
    },
    {
        "name": "No depression (9 channels)",
        "in_channels": 9,
        "description": "Remove LiDAR depression depth channel",
    },
    {
        "name": "No LiDAR (8 channels)",
        "in_channels": 8,
        "description": "Remove both elevation and depression channels",
    },
    {
        "name": "Single epoch (5 channels)",
        "in_channels": 5,
        "description": "Single NAIP epoch + elevation only",
    },
]

print("Ablation study configurations:")
for cfg in ablation_configs:
    print(f"  {cfg['name']}: {cfg['description']}")

print("\nNote: Running ablation requires custom composite creation")
print("that excludes specific bands. Implement per ablation config.")

---
## 5. Extended Training (100 epochs)

Final experiment with best hyperparameters and longer training.

In [ ]:
# Final experiment with best configuration
# Update learning_rate based on sweep results above
final_results = run_ppr_experiment(
    output_root="/content/ppr_final",
    overrides={
        "num_epochs": 100,
        "batch_size": 8,
        "learning_rate": 1e-3,  # Update with best LR from sweep
        "pre_downloaded_dem_tiles": [str(p) for p in DEM_TILE_PATHS],
        "pre_downloaded_naip": {
            2015: NAIP_2015,
            2017: NAIP_2017,
        },
    },
)

print("\nFinal experiment complete!")
table = format_results_table(final_results["results"])
print(table)

---
## 6. Results Analysis and Visualization

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

# Load results
results_path = final_results["output_path"]
with open(results_path) as f:
    saved = json.load(f)

results = saved["results"]
class_names = list(COWARDIN_CLASSES.values())

print(f"Loaded {len(results)} model results from {results_path}")

In [ ]:
# Plot 1: Overall accuracy and mIoU comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

model_names = [r["name"] for r in results]
oa_values = [r.get("overall_accuracy", 0) for r in results]
miou_values = [r.get("mean_iou", 0) for r in results]

palette = [plt.cm.tab10(i) for i in range(len(model_names))]

# OA bar chart
bars1 = axes[0].bar(model_names, oa_values, color=palette)
axes[0].set_ylabel("Overall Accuracy")
axes[0].set_title("Overall Accuracy by Model")
axes[0].set_ylim(0, 1)
for bar, val in zip(bars1, oa_values):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                 f"{val:.3f}", ha="center", va="bottom", fontweight="bold")

# mIoU bar chart
bars2 = axes[1].bar(model_names, miou_values, color=palette)
axes[1].set_ylabel("Mean IoU")
axes[1].set_title("Mean IoU by Model")
axes[1].set_ylim(0, 1)
for bar, val in zip(bars2, miou_values):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                 f"{val:.3f}", ha="center", va="bottom", fontweight="bold")

plt.tight_layout()
plt.savefig("/content/ppr_final/accuracy_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: /content/ppr_final/accuracy_comparison.png")

In [ ]:
# Plot 2: Per-class IoU comparison
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(class_names))
width = 0.35
palette = plt.cm.tab10.colors  # Dynamic color palette for any number of models

for i, r in enumerate(results):
    per_class_iou = r.get("per_class_iou", {})
    iou_values = [per_class_iou.get(str(j), per_class_iou.get(j, 0)) for j in range(len(class_names))]
    offset = (i - len(results)/2 + 0.5) * width
    bars = ax.bar(x + offset, iou_values, width, label=r["name"], color=palette[i % len(palette)])

ax.set_xlabel("Cowardin Class")
ax.set_ylabel("IoU")
ax.set_title("Per-Class IoU by Model")
ax.set_xticks(x)
ax.set_xticklabels(class_names, rotation=30, ha="right")
ax.legend()
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig("/content/ppr_final/per_class_iou.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: /content/ppr_final/per_class_iou.png")

In [ ]:
# Plot 3: Confusion matrix (if available)
for r in results:
    cm = r.get("confusion_matrix")
    if cm is not None:
        cm = np.array(cm)
        fig, ax = plt.subplots(figsize=(8, 7))

        # Normalize by row (true class)
        cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
        cm_norm = np.nan_to_num(cm_norm)

        im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
        ax.set_xticks(range(len(class_names)))
        ax.set_yticks(range(len(class_names)))
        ax.set_xticklabels(class_names, rotation=45, ha="right")
        ax.set_yticklabels(class_names)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True (NWI Reference)")
        ax.set_title(f"Confusion Matrix -- {r['name']}")

        # Add text annotations
        for i in range(len(class_names)):
            for j in range(len(class_names)):
                color = "white" if cm_norm[i, j] > 0.5 else "black"
                ax.text(j, i, f"{cm_norm[i,j]:.2f}",
                        ha="center", va="center", color=color, fontsize=9)

        plt.colorbar(im, ax=ax, shrink=0.8)
        plt.tight_layout()
        safe_name = r["name"].replace(" ", "_").replace("(", "").replace(")", "")
        plt.savefig(f"/content/ppr_final/confusion_matrix_{safe_name}.png",
                    dpi=150, bbox_inches="tight")
        plt.show()
        print(f"Saved confusion matrix for {r['name']}")
    else:
        print(f"No confusion matrix available for {r['name']}")

---
## 7. Prediction Visualization

In [ ]:
import rasterio
from matplotlib.colors import ListedColormap
from pathlib import Path

# Cowardin color scheme
cowardin_colors = [
    "#d9d9d9",  # 0: Upland (light gray)
    "#0077be",  # 1: Water (blue)
    "#7bc043",  # 2: Emergent (green)
    "#2d5016",  # 3: Forested (dark green)
    "#8b6914",  # 4: Scrub-Shrub (brown)
    "#daa520",  # 5: Other (goldenrod)
]
wetland_cmap = ListedColormap(cowardin_colors)

# Find prediction rasters
pred_dir = Path("/content/ppr_final/predictions")
if pred_dir.exists():
    pred_files = sorted(pred_dir.glob("pred_*.tif"))

    if not pred_files:
        print("No prediction files found in", pred_dir)
    else:
        fig, axes = plt.subplots(1, len(pred_files), figsize=(8 * len(pred_files), 8))
        if len(pred_files) == 1:
            axes = [axes]

        for ax, pred_file in zip(axes, pred_files):
            with rasterio.open(pred_file) as src:
                pred = src.read(1)

            im = ax.imshow(pred, cmap=wetland_cmap, vmin=0, vmax=5)
            ax.set_title(pred_file.stem.replace("pred_", ""), fontsize=10)
            ax.axis("off")

        # Add colorbar with class labels
        cbar = plt.colorbar(im, ax=axes, shrink=0.6, ticks=range(6))
        cbar.ax.set_yticklabels(list(COWARDIN_CLASSES.values()))

        plt.suptitle("Wetland Predictions -- PPR Study Area", fontsize=14)
        plt.tight_layout()
        plt.savefig("/content/ppr_final/prediction_maps.png", dpi=150, bbox_inches="tight")
        plt.show()
        print("Saved: /content/ppr_final/prediction_maps.png")
else:
    print(f"Prediction directory not found: {pred_dir}")
    print("Run the experiment first.")

---
## 8. Dynamics Visualization (2015 vs 2017)

In [ ]:
# Visualize wetland dynamics between 2015 and 2017
results_dir = Path("/content/ppr_final/results")
if results_dir.exists():
    dynamics_files = sorted(results_dir.glob("dynamics_*.tif"))

    if not dynamics_files:
        print("No dynamics rasters found. Need predictions from 2+ epochs.")
    else:
        dynamics_colors = [
            "#d9d9d9",  # 0: Stable non-wetland
            "#0077be",  # 1: Stable wetland
            "#ff4444",  # 2: Wetland loss
            "#44ff44",  # 3: Wetland gain
        ]
        dynamics_cmap = ListedColormap(dynamics_colors)
        dynamics_labels = ["Stable Non-Wetland", "Stable Wetland", "Wetland Loss", "Wetland Gain"]

        fig, axes = plt.subplots(1, len(dynamics_files), figsize=(8 * len(dynamics_files), 8))
        if len(dynamics_files) == 1:
            axes = [axes]

        for ax, dyn_file in zip(axes, dynamics_files):
            with rasterio.open(dyn_file) as src:
                dyn = src.read(1)

            im = ax.imshow(dyn, cmap=dynamics_cmap, vmin=0, vmax=3)
            ax.set_title(dyn_file.stem.replace("dynamics_", ""), fontsize=10)
            ax.axis("off")

        cbar = plt.colorbar(im, ax=axes, shrink=0.6, ticks=range(4))
        cbar.ax.set_yticklabels(dynamics_labels)

        plt.suptitle("Wetland Dynamics (2015-2017) -- PPR Study Area", fontsize=14)
        plt.tight_layout()
        plt.savefig("/content/ppr_final/dynamics_maps.png", dpi=150, bbox_inches="tight")
        plt.show()
        print("Saved: /content/ppr_final/dynamics_maps.png")
else:
    print("Results directory not found. Run the experiment first.")

---
## 9. Save Results to Google Drive

In [ ]:
# Mount Google Drive to persist results across sessions
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import shutil

# Copy results to Google Drive
drive_output = "/content/drive/MyDrive/wetland_experiment"
shutil.copytree("/content/ppr_final", drive_output, dirs_exist_ok=True)

print(f"Results saved to Google Drive: {drive_output}")
print("\nFiles saved:")
for f in sorted(Path(drive_output).rglob("*")):
    if f.is_file():
        size_mb = f.stat().st_size / 1e6
        print(f"  {f.relative_to(drive_output)} ({size_mb:.1f} MB)")

---
## 10. Summary Table for Paper

Final formatted results table ready for the manuscript.

In [ ]:
# Load final results and print paper-ready table
with open("/content/ppr_final/results/experiment_results.json") as f:
    final = json.load(f)

table = format_results_table(final["results"])
print("Table for paper (Markdown):")
print("="*80)
print(table)
print("="*80)

# Also print LaTeX version
print("\n\nLaTeX table (manual conversion):")
print("-"*80)
for r in final["results"]:
    oa = r.get("overall_accuracy", 0)
    miou = r.get("mean_iou", 0)
    per_class_iou = r.get("per_class_iou", {})
    iou_vals = " & ".join(
        f"{per_class_iou.get(str(i), per_class_iou.get(i, 0)):.3f}"
        for i in range(6)
    )
    print(f"{r['name']} & {oa:.3f} & {miou:.3f} & {iou_vals} \\\\")

---

## Notes

- **3DEP Timeout:** If DEM download fails, use Section 1 to upload pre-downloaded tiles from [USGS National Map](https://apps.nationalmap.gov/downloader/)
- **Troubleshooting OOM:** Reduce `batch_size` to 4 or `tile_size` to 128
- **Colab disconnects:** Results are saved to Google Drive at Section 9
- **Restarting:** Re-run Sections 0-1, then skip to the section you need
- **Data caching:** Downloaded data is in `/content/ppr_*/naip/`, `/content/ppr_*/dem.tif`, etc. Reuse across runs by copying to a shared location.

### Expected Output Structure
```
ppr_final/
  naip/           -- Downloaded NAIP imagery
  composites/     -- Spectral indices, composites, weak labels
  tiles/          -- Training tiles (256x256)
  models/         -- Trained model checkpoints
  predictions/    -- Prediction rasters per model/epoch
  results/        -- experiment_results.json, dynamics maps
```